### （0）安装依赖

In [1]:
#@!uv pip install -U vllm modelscope transformers accelerate datasets trl peft scikit-learn pandas tqdm torchvision --no-cache -i https://mirrors.cloud.tencent.com/pypi/simple/ --extra-index-url https://wheels.vllm.ai/rocm/

### （1）导入依赖和全局配置

In [2]:
import os
import re
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict, ClassLabel, load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from modelscope import snapshot_download
from modelscope.hub.snapshot_download import dataset_snapshot_download

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")

# -----------------------------
# 基础配置
# -----------------------------
# 魔搭上的模型 ID（Gemma 4 E4B-it 在 ModelScope 上的官方仓库,instruction-tuned 版本,
# 仓库内自带官方 chat_template.jinja,无需手动处理 chat template)。
# 仓库地址: https://www.modelscope.cn/models/google/gemma-4-E4B-it
MODELSCOPE_MODEL_ID = "google/gemma-4-E4B-it"

# 魔搭上的数据集 ID（dair-ai/emotion 在 ModelScope 上的官方镜像）。
MODELSCOPE_DATASET_ID = "AI-ModelScope/emotion"

# 微调输出目录
OUTPUT_DIR = "./gemma4-it-emotion-lora-ms-single-gpu"

# 数据量控制。先用小数据跑通，确认没问题后再加大。
TRAIN_LIMIT = 4000
VALIDATION_LIMIT = 400
TEST_LIMIT = 400
EVAL_LIMIT = 400

SEED = 42
MODEL_DTYPE = torch.bfloat16
BF16 = True
FP16 = False

SYSTEM_PROMPT = """You are an emotion classification assistant.
Read the user's text and answer with exactly one label.
Only choose from: sadness, joy, love, anger, fear, surprise.
Return only the label and nothing else."""

LABEL_PATTERN = re.compile(r"(sadness|joy|love|anger|fear|surprise)", re.IGNORECASE)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./datasets", exist_ok=True)

print("torch version:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))

/mnt/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.12.0+git6bbd260
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
current device: 0
device name: 


In [5]:
def setup_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

setup_seed(SEED)

### （2）从魔搭 ModelScope 下载 Gemma 模型

In [6]:
from pathlib import Path
MODELSCOPE_MODEL_ID = "google/gemma-4-E4B-it"
print("Downloading model from ModelScope...")
print("ModelScope model id:", MODELSCOPE_MODEL_ID)

new_cache = Path("/root/tmp/models")
new_cache.mkdir(parents=True, exist_ok=True)
model_dir = snapshot_download(
    MODELSCOPE_MODEL_ID,
    cache_dir=str(new_cache),
)

print("Downloaded model dir:", model_dir)

# 后续统一使用本地路径加载
LOCAL_MODEL_DIR = model_dir
print("Downloaded model dir:", LOCAL_MODEL_DIR)

ModelScope model id: google/gemma-4-E4B-it


2026-10-26 18:08:06,689 | INFO    | modelscope_hub.download | Downloading 10 files from google/gemma-4-E4B-it@master
Downloading: 100%|██████████| 10/10 [00:12<00:00,  1.24s/file]

Downloaded model dir: /root/tmp/models/models/google--gemma-4-E4B-it/snapshots/master
Downloaded model dir: /root/tmp/models/models/google--gemma-4-E4B-it/snapshots/master


### （3）加载Emotion数据集

In [7]:
import glob

EMOTION_LABEL_NAMES = ["sadness", "joy", "love", "anger", "fear", "surprise"]


# 直接把魔搭上的数据集仓库(parquet 文件)整体下载到本地,然后用 datasets 库从本地 parquet 加载。
# 不走 MsDataset.load -> datasets.load_dataset 的桥接路径,可以规避 modelscope 与 datasets 之间
# `as_dataset() got an unexpected keyword argument 'verification_mode'` 这类版本错配错误。
print("Downloading dataset from ModelScope...")
print("ModelScope dataset id:", MODELSCOPE_DATASET_ID)

dataset_dir = dataset_snapshot_download(
    MODELSCOPE_DATASET_ID,
    cache_dir="./datasets",
)
print("Downloaded dataset dir:", dataset_dir)


def _parquet_files_for(split_name: str):
    pattern = os.path.join(dataset_dir, "data", f"{split_name}-*.parquet")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No parquet files matched pattern: {pattern}. "
            f"Please check the dataset repo layout under {dataset_dir}."
        )
    return files


raw_dataset = load_dataset(
    "parquet",
    data_files={
        "train": _parquet_files_for("train"),
        "validation": _parquet_files_for("validation"),
        "test": _parquet_files_for("test"),
    },
)

# 从 parquet 加载时,label 字段类型会退化成普通整数,这里显式 cast 成 ClassLabel,
# 这样后续 `dataset["train"].features["label"].names` 和原始 HF 版接口完全一致。
for split_name in list(raw_dataset.keys()):
    if not isinstance(raw_dataset[split_name].features.get("label"), ClassLabel):
        raw_dataset[split_name] = raw_dataset[split_name].cast_column(
            "label", ClassLabel(names=EMOTION_LABEL_NAMES)
        )

print("Raw dataset:", raw_dataset)


def maybe_limit(split, limit):
    split = split.shuffle(seed=SEED)
    if limit is None:
        return split
    return split.select(range(min(limit, len(split))))


dataset = DatasetDict({
    "train": maybe_limit(raw_dataset["train"], TRAIN_LIMIT),
    "validation": maybe_limit(raw_dataset["validation"], VALIDATION_LIMIT),
    "test": maybe_limit(raw_dataset["test"], TEST_LIMIT),
})

label_names = dataset["train"].features["label"].names
VALID_LABELS = set(label_names)
ALL_EVAL_LABELS = label_names + ["INVALID"]

print(dataset)
print("label_names:", label_names)
print("example:", dataset["train"][0])

ModelScope dataset id: AI-ModelScope/emotion


2026-10-26 18:08:25,769 | INFO    | modelscope_hub.legacy_api | Listing AI-ModelScope/emotion@master Root=/ ...
2026-10-26 18:08:26,104 | INFO    | modelscope_hub.legacy_api | Listing AI-ModelScope/emotion Root=/: page 1, 7 entries
2026-10-26 18:08:26,104 | INFO    | modelscope_hub.download | Downloading 6 files from AI-ModelScope/emotion@master
Downloading: 100%|██████████| 6/6 [00:00<00:00, 408.09file/s]


Downloaded dataset dir: datasets/datasets/AI-ModelScope--emotion/snapshots/master
Raw dataset: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
})
label_names: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
example: {'text': 'while cycling in the country', 'label': 4}


###  构造 prompt-completion 格式数据

In [8]:
def to_prompt_completion(example):
    text = example["text"]
    label = label_names[example["label"]]
    user_content = f"Classify the emotion of this text:\n\n{text}"
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": label},
        ],
    }

sft_dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names,
)

print(sft_dataset)
print(sft_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 400
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 400
    })
})
{'prompt': [{'role': 'system', 'content': "You are an emotion classification assistant.\nRead the user's text and answer with exactly one label.\nOnly choose from: sadness, joy, love, anger, fear, surprise.\nReturn only the label and nothing else."}, {'role': 'user', 'content': 'Classify the emotion of this text:\n\nwhile cycling in the country'}], 'completion': [{'role': 'assistant', 'content': 'fear'}]}


加载 tokenizer 和基础模型

In [9]:
print("Loading tokenizer from:", LOCAL_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_DIR,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

# `google/gemma-4-E4B-it` 的 tokenizer 通常会自带 chat_template。
# 若缺失（缓存不完整等），从同一魔搭仓库拉取官方 chat_template.jinja 注入（权重已在上面整仓下载时可跳过额外拉取）。
TEMPLATE_SOURCE_MODEL_ID = "google/gemma-4-E4B-it"

def _load_official_gemma_chat_template() -> str:
    """从 gemma-4-E4B-it 仓库下载官方 chat_template.jinja 并返回字符串。

    主路径:modelscope.snapshot_download(allow_file_pattern=["chat_template.jinja"])
    兜底:ModelScope raw file API 直接 HTTP GET
    """
    try:
        template_dir = snapshot_download(
            TEMPLATE_SOURCE_MODEL_ID,
            cache_dir="./models",
            allow_file_pattern=["chat_template.jinja"],
        )
        path = os.path.join(template_dir, "chat_template.jinja")
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return f.read()
    except Exception as e:
        print("snapshot_download(allow_file_pattern) failed, fallback to HTTP. err =", e)

    import urllib.request
    url = (
        "https://www.modelscope.cn/api/v1/models/"
        f"{TEMPLATE_SOURCE_MODEL_ID}/repo?Revision=master&FilePath=chat_template.jinja"
    )
    with urllib.request.urlopen(url, timeout=60) as resp:
        return resp.read().decode("utf-8")


if not getattr(tokenizer, "chat_template", None):
    print(f"Loading official chat_template.jinja from {TEMPLATE_SOURCE_MODEL_ID} ...")
    tokenizer.chat_template = _load_official_gemma_chat_template()
    print("Loaded official chat_template, length =", len(tokenizer.chat_template))
else:
    print("tokenizer.chat_template already set, leaving as-is.")

# 自检:跑一次 apply_chat_template,确保模板可用。
_probe = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
print("chat_template probe output:\n" + _probe)

print("Loading base model from:", LOCAL_MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("HIP version:", getattr(torch.version, "hip", None))

base_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

base_model.to(device)

base_model.config.use_cache = False
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.bos_token_id = tokenizer.bos_token_id
base_model.config.eos_token_id = tokenizer.eos_token_id

base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.bos_token_id = tokenizer.bos_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Base model loaded.")
print("Base model device:", next(base_model.parameters()).device)

Loading tokenizer from: /root/tmp/models/models/google--gemma-4-E4B-it/snapshots/master


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pad_token: <pad>
eos_token: <eos>
tokenizer.chat_template already set, leaving as-is.
chat_template probe output:
<bos><|turn>system
You are a helpful assistant.<turn|>
<|turn>user
Hello<turn|>
<|turn>model

Loading base model from: /root/tmp/models/models/google--gemma-4-E4B-it/snapshots/master
Using device: cuda
HIP version: 7.2.53211


Loading weights: 100%|██████████| 2076/2076 [00:00<00:00, 11660.80it/s]


Base model loaded.
Base model device: cuda:0


推理辅助函数

In [10]:
def extract_label(raw_text: str) -> str:
    raw_text = raw_text.strip().lower()
    match = LABEL_PATTERN.search(raw_text)
    if match:
        return match.group(1)

    tokens = raw_text.split()
    if not tokens:
        return "INVALID"

    return tokens[0].strip(".,!?:;\"'()[]{}")


def generate_label(model, tokenizer, user_text: str, system_prompt: str = SYSTEM_PROMPT, max_new_tokens: int = 4) -> str:
    user_content = f"Classify the emotion of this text:\n\n{user_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

    device = next(model.parameters()).device

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[-1]
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    raw_pred = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return extract_label(raw_pred)


def predict_emotion(text: str, model=None):
    model = model or base_model
    return generate_label(model, tokenizer, text)

predict_emotion("I feel so happy and excited today!")

'joy'

评估函数

In [11]:
def evaluate_model(model, tokenizer, split="test", limit=EVAL_LIMIT):
    y_true, y_pred, rows = [], [], []
    raw_source = dataset[split]

    if limit is not None:
        raw_source = raw_source.select(range(min(limit, len(raw_source))))

    model.eval()

    for ex in tqdm(raw_source, desc=f"Evaluating {split}", leave=False):
        true_label = label_names[ex["label"]]
        raw_pred_label = generate_label(model, tokenizer, ex["text"], SYSTEM_PROMPT)
        pred_label = raw_pred_label if raw_pred_label in VALID_LABELS else "INVALID"

        y_true.append(true_label)
        y_pred.append(pred_label)
        rows.append({
            "text": ex["text"],
            "true_label": true_label,
            "pred_label": pred_label,
            "raw_pred_label": raw_pred_label,
            "correct": true_label == pred_label,
        })

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=label_names, average="macro", zero_division=0),
        "invalid_predictions": sum(1 for p in y_pred if p == "INVALID"),
        "evaluated_examples": len(y_true),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=label_names,
        output_dict=True,
        zero_division=0,
    )

    return metrics, report, pd.DataFrame(rows)


def confusion_matrix_df(pred_df):
    return pd.DataFrame(
        confusion_matrix(pred_df["true_label"], pred_df["pred_label"], labels=ALL_EVAL_LABELS),
        index=ALL_EVAL_LABELS,
        columns=ALL_EVAL_LABELS,
    )

微调前评估：先评估基础模型的分类能力，作为后面 LoRA 微调后的对比基线。

In [12]:
pre_metrics, pre_report, pre_preds = evaluate_model(base_model, tokenizer, split="test", limit=EVAL_LIMIT)
pre_metrics

{'accuracy': 0.6225,
 'macro_f1': 0.4750600720817734,
 'invalid_predictions': 2,
 'evaluated_examples': 400}

In [13]:
pd.DataFrame(pre_report).transpose()

,precision,recall,f1-score,support
sadness,0.551136,0.850877,0.668966,114.0
joy,0.766667,0.709877,0.737179,162.0
love,0.333333,0.192308,0.243902,26.0
anger,0.625000,0.319149,0.422535,47.0
fear,0.666667,0.333333,0.444444,36.0
surprise,0.333333,0.333333,0.333333,15.0
micro avg,0.625628,0.622500,0.624060,400.0
macro avg,0.546023,0.456480,0.475060,400.0
weighted avg,0.635178,0.622500,0.607214,400.0


In [14]:
confusion_matrix_df(pre_preds)

,sadness,joy,love,anger,fear,surprise,INVALID
sadness,97,10,1,2,2,1,1
joy,25,115,7,6,2,6,1
love,7,14,5,0,0,0,0
anger,26,3,1,15,2,0,0
fear,16,4,0,1,12,3,0
surprise,5,4,1,0,0,5,0
INVALID,0,0,0,0,0,0,0


配置 LoRA

In [18]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

In [16]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    num_train_epochs=1,

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True,

    bf16=BF16,
    fp16=FP16,
    tf32=False,

    max_length=256,
    packing=False,
    completion_only_loss=True,

    remove_unused_columns=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

开始 LoRA 微调

In [17]:
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer,
)

trainable_params = 0
total_params = 0
trainable_param_names = []

for name, param in trainer.model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        trainable_param_names.append(name)

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters were attached. Check target_modules before training.")

print(f"Trainable LoRA parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable ratio: {100 * trainable_params / total_params:.4f}%")
print("Example trainable parameters:")
print(trainable_param_names[:20])

train_result = trainer.train()

trainer.model.eval()
trainer.model.config.use_cache = True
train_result

Trainable LoRA parameters: 50,499,584
Total parameters: 7,991,600,416
Trainable ratio: 0.6319%
Example trainable parameters:
['base_model.model.model.vision_tower.patch_embedder.input_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.patch_embedder.input_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.linear.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.linear.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.linear.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.linear.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.linear.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.linear.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.o_proj.linear.lora_A.def

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,0.571625,0.404875,0.455759,0.845625,37055.000000
50,0.337790,0.304260,0.232992,0.895833,74411.000000
75,0.206285,0.225538,0.138760,0.929792,111380.000000
100,0.149392,0.181775,0.143737,0.936875,148627.000000
125,0.145339,0.159709,0.132591,0.941250,185890.000000
150,0.123479,0.148414,0.114916,0.945208,223212.000000
175,0.119190,0.137781,0.069864,0.952917,260384.000000
200,0.106114,0.114019,0.100465,0.953542,297687.000000
225,0.108883,0.093608,0.080593,0.962917,334798.000000
250,0.104989,0.087279,0.087406,0.964167,372021.000000


TrainOutput(global_step=250, training_loss=0.31420770835876466, metrics={'train_runtime': 1115.5292, 'train_samples_per_second': 3.586, 'train_steps_per_second': 0.224, 'total_flos': 1.134650820797568e+16, 'train_loss': 0.31420770835876466, 'epoch': 1.0})

保存 LoRA adapter 和 tokenizer

In [19]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print("Saved adapter and tokenizer to:", OUTPUT_DIR)

Saved adapter and tokenizer to: ./gemma4-it-emotion-lora-ms-single-gpu


微调后评估

In [20]:
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True

post_metrics, post_report, post_preds = evaluate_model(ft_model, tokenizer, split="test", limit=EVAL_LIMIT)
post_metrics

{'accuracy': 0.92,
 'macro_f1': 0.8704364652946213,
 'invalid_predictions': 0,
 'evaluated_examples': 400}

In [25]:
pd.DataFrame(post_report).transpose()

,precision,recall,f1-score,support
sadness,0.923077,0.947368,0.935065,114.00
joy,0.934132,0.962963,0.948328,162.00
love,0.791667,0.730769,0.760000,26.00
anger,1.000000,0.872340,0.931818,47.00
fear,0.871795,0.944444,0.906667,36.00
surprise,0.833333,0.666667,0.740741,15.00
accuracy,0.920000,0.920000,0.920000,0.92
macro avg,0.892334,0.854092,0.870436,400.00
weighted avg,0.920070,0.920000,0.918833,400.00


In [26]:
confusion_matrix_df(post_preds)

,sadness,joy,love,anger,fear,surprise,INVALID
sadness,108,3,1,0,2,0,0
joy,2,156,4,0,0,0,0
love,2,5,19,0,0,0,0
anger,5,1,0,41,0,0,0
fear,0,0,0,0,34,2,0
surprise,0,2,0,0,3,10,0
INVALID,0,0,0,0,0,0,0


对比微调前后效果

In [27]:
comparison_df = pd.DataFrame([
    {"stage": "pre_finetuning", **pre_metrics},
    {"stage": "post_finetuning", **post_metrics},
])

comparison_df

,stage,accuracy,macro_f1,invalid_predictions,evaluated_examples
0,pre_finetuning,0.6225,0.475060,2,400
1,post_finetuning,0.9200,0.870436,0,400


In [29]:
merged_examples = pre_preds.copy()
merged_examples = merged_examples.rename(columns={
    "pred_label": "pre_pred",
    "correct": "pre_correct",
    "raw_pred_label": "pre_raw_pred_label",
})

merged_examples["post_pred"] = post_preds["pred_label"]
merged_examples["post_raw_pred_label"] = post_preds["raw_pred_label"]
merged_examples["post_correct"] = post_preds["correct"]

changed_predictions = merged_examples[merged_examples["pre_pred"] != merged_examples["post_pred"]]
changed_predictions.head(20)

,text,true_label,pre_pred,pre_raw_pred_label,pre_correct,post_pred,post_raw_pred_label,post_correct
2,i feel is that the most likeable characters ar...,joy,sadness,sadness,False,joy,joy,True
5,im feeling and if ive liked being pregnant,love,joy,joy,False,love,love,True
7,i used to be able to hang around talk with the...,anger,sadness,sadness,False,anger,anger,True
8,i don t have the feeling of divine vibrations,joy,sadness,sadness,False,joy,joy,True
9,i vented my feelings towards the pathetic excu...,sadness,anger,anger,False,sadness,sadness,True
11,i get the feeling that this could be dangerous,anger,fear,fear,False,anger,anger,True
12,i feel like some of you have pains and you can...,joy,sadness,sadness,False,love,love,False
13,i cant write a review for a book i adore unles...,love,joy,joy,False,love,love,True
14,i mentioned in my last blog that i have starte...,fear,sadness,sadness,False,fear,fear,True
16,i was wondering if you will focus on the probl...,sadness,anger,anger,False,sadness,sadness,True


手动测试微调后的模型

In [30]:
def predict_emotion_ft(user_text: str) -> str:
    return generate_label(ft_model, tokenizer, user_text, SYSTEM_PROMPT)

test_texts = [
    "I feel completely heartbroken and alone.",
    "This is the best day of my life!",
    "I am really scared about what might happen tomorrow.",
    "I can't believe they remembered my birthday!",
    "I am so angry that nobody listened to me.",
    "I really love spending time with my family.",
]

for text in test_texts:
    print(text, "=>", predict_emotion_ft(text))

I feel completely heartbroken and alone. => sadness
This is the best day of my life! => joy
I am really scared about what might happen tomorrow. => fear
I can't believe they remembered my birthday! => joy
I am so angry that nobody listened to me. => anger
I really love spending time with my family. => love


In [31]:
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "gemma4_emotion_before_after_metrics.csv"), index=False)
merged_examples.to_csv(os.path.join(OUTPUT_DIR, "gemma4_emotion_prediction_examples.csv"), index=False)
changed_predictions.to_csv(os.path.join(OUTPUT_DIR, "gemma4_emotion_changed_predictions.csv"), index=False)
pre_preds.to_csv(os.path.join(OUTPUT_DIR, "pre_finetuning_predictions.csv"), index=False)
post_preds.to_csv(os.path.join(OUTPUT_DIR, "post_finetuning_predictions.csv"), index=False)

pd.DataFrame(pre_report).transpose().to_csv(os.path.join(OUTPUT_DIR, "pre_finetuning_classification_report.csv"))
pd.DataFrame(post_report).transpose().to_csv(os.path.join(OUTPUT_DIR, "post_finetuning_classification_report.csv"))
confusion_matrix_df(pre_preds).to_csv(os.path.join(OUTPUT_DIR, "pre_finetuning_confusion_matrix.csv"))
confusion_matrix_df(post_preds).to_csv(os.path.join(OUTPUT_DIR, "post_finetuning_confusion_matrix.csv"))

print("Saved all outputs to:", OUTPUT_DIR)

Saved all outputs to: ./gemma4-it-emotion-lora-ms-single-gpu


In [34]:
# 可选运行：重新加载 LoRA adapter
# 为避免当前 notebook 占用显存，建议重启 kernel 后单独运行本节。

RUN_RELOAD_TEST = True

if RUN_RELOAD_TEST:
    reload_tokenizer = AutoTokenizer.from_pretrained(
        OUTPUT_DIR,
        use_fast=True,
        trust_remote_code=True,
    )
    if reload_tokenizer.pad_token is None:
        reload_tokenizer.pad_token = reload_tokenizer.eos_token

    reload_base_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_DIR,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    reload_model = PeftModel.from_pretrained(
        reload_base_model,
        OUTPUT_DIR,
    )
    reload_model.eval()

    print(generate_label(reload_model, reload_tokenizer, "I feel completely heartbroken and alone."))

Loading weights: 100%|██████████| 2076/2076 [00:00<00:00, 9778.67it/s] 


sadness
